In [1]:
print("ok")

ok


In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
import os
from pinecone import Pinecone, ServerlessSpec

c:\Users\maaz7\localllama\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\maaz7\AppData\Local\Temp\ipykernel_32608\238595301.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    )

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10492.60it/s]


In [6]:
load_dotenv()
pinecone_api_key=os.getenv("PINECONE_API_KEY")


In [7]:
pc=Pinecone(api_key=pinecone_api_key)

In [8]:
index_name = "medical-chatbot-rag"


if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension =1024,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )

    )

In [9]:
index=pc.Index("medical-chatbot-rag")

In [10]:
def load_pdf(data):
    loader = DirectoryLoader(data,
                     glob="*.pdf",
                     loader_cls=PyPDFLoader
                     )
    documents = loader.load()
    return documents

In [11]:
extracted_data = load_pdf("data/")

In [29]:
# extracted_data

Create text chunks

In [30]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap = 20
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [31]:
text_chunks = text_split(extracted_data)
print(len(text_chunks))

14618


In [34]:
len(embeddings.embed_query("Hello World"))

1024

In [ ]:
# docsearch = PineconeVectorStore.from_texts([t.page_content for t in text_chunks], embeddings, index_name = index_name)

In [12]:
docsearch=PineconeVectorStore(index_name=index_name, embedding=embeddings)
query = "What are alergies?"

docs = docsearch.similarity_search(query, k=3)

print("Result: ", docs)

Result:  [Document(id='c7fa2248-0ac3-49d8-8b84-2f5b2e9692b3', metadata={}, page_content='Eczema—Inflammation of the skin with itching\nand a rash. The rash may have blisters that ooze\nand form crusts.\nPimple—A small, red swelling of the skin.\nPsoriasis —A skin disease in which people have\nitchy, scaly, red patches on the skin.\nPus—Thick, whitish or yellowish fluid that forms\nin infected tissue.\nTriglyceride —A substance formed in the body\nfrom fat in the diet.\nand stinging, and a warm feeling to the skin. These prob-\nlems usually go away as the body adjusts to the drug and'), Document(id='2d0ecaa5-543f-4c3a-9af1-dc213be765dd', metadata={}, page_content='Eczema \nEczema is an allergic reaction that manifests as dry, itchy patches of skin that resemble rashes (Figure 5.21). It may \nbe accompanied by swelling of the skin, flaking, and in severe cases, bleeding. Many who suffer from eczema have \nantibodies against dust mites in their blood, but the link between eczema and aller

In [13]:
docs

[Document(id='c7fa2248-0ac3-49d8-8b84-2f5b2e9692b3', metadata={}, page_content='Eczema—Inflammation of the skin with itching\nand a rash. The rash may have blisters that ooze\nand form crusts.\nPimple—A small, red swelling of the skin.\nPsoriasis —A skin disease in which people have\nitchy, scaly, red patches on the skin.\nPus—Thick, whitish or yellowish fluid that forms\nin infected tissue.\nTriglyceride —A substance formed in the body\nfrom fat in the diet.\nand stinging, and a warm feeling to the skin. These prob-\nlems usually go away as the body adjusts to the drug and'),
 Document(id='2d0ecaa5-543f-4c3a-9af1-dc213be765dd', metadata={}, page_content='Eczema \nEczema is an allergic reaction that manifests as dry, itchy patches of skin that resemble rashes (Figure 5.21). It may \nbe accompanied by swelling of the skin, flaking, and in severe cases, bleeding. Many who suffer from eczema have \nantibodies against dust mites in their blood, but the link between eczema and allergy to du

In [14]:
promt_template = """
Use the following information to answer the user's question.
if you don't know the answer do not make it up just say you do not know.

Context: {context}
Question: {question}

only return the perfect answer below and nothing else

Answer: 


"""

In [15]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template = promt_template
)
chain_type_kwargs = {"prompt":prompt}


In [16]:
llm = ChatOllama(
    model = "gemma3:4b",
    max_new_tokens = 512,
    temperature=0.8
)

In [17]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever = docsearch.as_retriever(search_kwargs={'k':2}),
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs
)

In [ ]:
while True:
    user_input= input(f"Input Prompt:")
    result=qa({"query": user_input})
    print("response:", result["result"])

C:\Users\maaz7\AppData\Local\Temp\ipykernel_32608\1670928500.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result=qa({"query": user_input})


response: A block in the blood supply to the heart, resulting in what is commonly called a heart attack.
response: I do not know. The text suggests seeking a fast-acting ACE inhibitor to relieve attacks and advises getting up gradually to lessen lightheadedness, dizziness, or fainting. However, it doesn’t provide comprehensive instructions for handling a heart attack itself.
response: I do not know.
response: I do not know.
